# GeoSPARQL 1.1 Tutorial

## Installing requirements

In [1]:
!pip install -r requirements.txt --break-system-packages
!pip install ipyleaflet --break-system-packages
!jupyter labextension install jupyter-leaflet

import rdflib
import shapely.geometry
from shapely import wkt
from shapely.ops import transform
from rdflib import Graph, Literal
import os
import itertools
from shapely.testing import assert_geometries_equal
from geosparql.geosparql import LiteralUtils
from geosparql.geosparql_aggregates import processLiteralTypeToGeom
from IPython.display import display, HTML
import ipywidgets as W
from ipyleaflet import Map, WKTLayer, FullScreenControl, Popup#, Tooltip

mapcenter=(34.1,-83.2)
zoomlevel=10
GEO = "http://www.opengis.net/ont/geosparql#"
GEOF = "http://www.opengis.net/def/function/geosparql/"
GEOFEXT = "http://www.opengis.net/def/function/geosparql/ext/"

def getHTMLStringFromQueryResult(qres):
    res="<table><tr><th>Variable</th><th>Value</th></tr>"
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in resultlist:
        for item in row:
            if isinstance(row[item],Literal) and row[item].datatype!=None:
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item].datatype)+"\">"+str(row[item]).strip()+"</a></td></tr>"
            elif isinstance(row[item],URIRef):
                res+="<tr><td>"+str(item)+"</td><td><a href=\""+str(row[item])+"\">"+str(row[item]).strip()+"</a></td></tr>"
            else:
                res+="<tr><td>"+str(item)+"</td><td>"+str(row[item]).strip()+"</td></tr>"
    res+="</table>"
    return res

def getMapFromWKTResult(qres,rows=[],lmap=None):
    layers=[]
    lastcentroid=shapely.geometry.Point(mapcenter[0],mapcenter[1])
    if qres is not None and len(qres.bindings)>0:
        resultlist=[{str(k): v for k, v in i.items()} for i in qres.bindings]
    for row in rows:
        if row in resultlist[0]:
            popup="<ul>"
            for rowres in resultlist:
                for item in rowres:
                    if isinstance(rowres[item],Literal) and rowres[item].datatype!=None:
                        popup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item].datatype)+"\">"+str(rowres[item]).strip()+"</a></li>"
                    elif isinstance(rowres[item],URIRef):
                        poup+="<li><b>"+str(item)+":</b> <a href=\""+str(rowres[item])+"\">"+str(rowres[item]).strip()+"</a></li>"
                    else:
                        popup+="<li><b>"+str(item)+":</b> "+str(rowres[item]).strip()+"</li>"
            popup+="</ul>"
            toprocess=resultlist[0][row].strip()
            if toprocess.startswith("<http"):
                toprocess=toprocess[toprocess.find(">")+1:]
            geom = wkt.loads(toprocess)
            if not geom.centroid.is_empty:
                lastcentroid=geom.centroid
            nlayer=WKTLayer(wkt_string=geom.wkt,hover_style={"fillColor": "red"})
            msg=W.HTML()
            msg.value=popup
            nlpopup=Popup(
                location=[lastcentroid.y,lastcentroid.x],
                child=msg,
                close_button=True,
                auto_close=False,
                close_on_escape_key=False
            )
            layers.append(nlayer)
            layers.append(nlpopup)
            nlpopup.close_popup()
    if lmap==None:
        lmap=Map(center=(lastcentroid.y,lastcentroid.x), zoom=zoomlevel)
        control = FullScreenControl()
        lmap.add(control)
    for lay in layers:
        lmap.add(lay)
    return lmap    

def processQueryResults(qres,rows=[],lmap=None):
    display(HTML(getHTMLStringFromQueryResult(qres)))
    if rows!=[]:
        lmap=getMapFromWKTResult(qres,rows,lmap)
    return lmap

g=Graph()
g.parse("tests/testdata.ttl")
print(len(g))

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:54: UserWarning: An error occurred.
  warnings.warn("An error occurred.")
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:55: UserWarning: PermissionError: [Errno 13] Permission denied: '/usr/local/share/jupyter/lab/extensions/jupyter-leaflet-0.20.0.tgz'
  warnings.warn(msg[-1].strip())
/usr/local/lib/python3.13/dist-packages/jupyterlab/debuglog.py:56: UserWarning: See the log file for details: /tmp/jupyterlab-debug-q02lisu1.log


## GeoSPARQL 1.1 Serialization Functions

### [geof:asGeoJSON](http://www.opengis.net/def/function/geosparql/asGeoJSON) Function

Retrieves a GeoJSON serialization of a GeoSPARQL geometry.

If the input geometry is not in the WGS84 coordinate system, the geometry will be converted to WGS84.

In [2]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?geojson
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGeoJSON(?aLiteral) as ?geojson)
}
"""),["aLiteral"],None)

GEOMTOLIT: POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
geojson,"{""type"":""Polygon"",""coordinates"":[[[-83.6,34.1],[-83.2,34.1],[-83.2,34.5],[-83.6,34.5],[-83.6,34.1]]]}"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asGML](http://www.opengis.net/def/function/geosparql/asGML) Function

In [3]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?gml
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asGML(?aLiteral) as ?gml)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:asKML](http://www.opengis.net/def/function/geosparql/asKML) Function

In [4]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?kml
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:asKML(?aLiteral) as ?kml)
}
"""),["aLiteral"],None)

GEOMTOLIT: POLYGON ((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
kml,"-83.6,34.1 -83.2,34.1 -83.2,34.5 -83.6,34.5 -83.6,34.1"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Non-topological query functions

### [geof:area](http://www.opengis.net/def/function/geosparql/area) Function

In [5]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?area
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:area(?aLiteral,"http://qudt.org/vocab/unit/AC"^^xsd:anyURI) as ?area)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
area,593079.4121828682


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:boundingCircle](http://www.opengis.net/def/function/geosparql/boundingCircle) Function

In [23]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?bcirc
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:boundingCircle(?aLiteral) as ?bcirc)
}
"""),["aLiteral","bcirc"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:centroid](http://www.opengis.net/def/function/geosparql/centroid) Function

In [6]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?centroid
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:centroid(?aLiteral) as ?centroid)
}
"""),["aLiteral","centroid"],None)

GEOMTOLIT: POINT (-83.39999999999999 34.300000000000004)
<class 'shapely.geometry.point.Point'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
centroid,POINT (-83.39999999999999 34.300000000000004)


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:concaveHull](http://www.opengis.net/def/function/geosparql/concaveHull) Function

In [22]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?chull
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:concaveHull(?aLiteral) as ?chull)
}
"""),["aLiteral","chull"],None)

GEOMTOLIT: POLYGON ((-83.2 34.1, -83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
chull,"POLYGON ((-83.2 34.1, -83.6 34.1, -83.6 34.5, -83.2 34.5, -83.2 34.1))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:geometryN](http://www.opengis.net/def/function/geosparql/geometryN) Function

In [24]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?geomN
WHERE {
  BIND("MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"^^geo:wktLiteral AS ?aLiteral)
  BIND(geof:geometryN(?aLiteral,"1"^^xsd:integer) AS ?geomN)
}
"""),["aLiteral"],None)

MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))
POLYGON ((4 3, 6 3, 6 1, 4 1, 4 3))
GEOMTOLIT: POLYGON ((4 3, 6 3, 6 1, 4 1, 4 3))
<class 'shapely.geometry.polygon.Polygon'>


Variable,Value
aLiteral,"MULTIPOLYGON (((1 1, 1 3, 3 3, 3 1, 1 1)), ((4 3, 6 3, 6 1, 4 1, 4 3)))"
geomN,"POLYGON ((4 3, 6 3, 6 1, 4 1, 4 3))"


Map(center=[2.0, 3.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

### [geof:is3D](http://www.opengis.net/def/function/geosparql/is3D) Function

In [7]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?n_wkt ?k_wkt ?isN3D ?isK3D {
<http://example.org/ApplicationSchema#NExactGeom> geo:asWKT ?n_wkt .
BIND(geof:is3D(?n_wkt) AS ?isN3D)
<http://example.org/ApplicationSchema#KExactGeom> geo:asWKT ?k_wkt .
BIND(geof:is3D(?k_wkt) AS ?isK3D)
}
"""),["n_wkt","k_wkt"],None)

Variable,Value
n_wkt,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
k_wkt,"Polygon((-77.089005 38.913574,-77.029953 38.913574,-77.029953 38.886321,-77.089005 38.886321,-77.089005 38.913574))"


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:isEmpty](http://www.opengis.net/def/function/geosparql/isEmpty) Function

In [8]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?i_wkt ?k_wkt (xsd:boolean(?isIEmptyr) as ?isIEmpty) (xsd:boolean(?isKEmptyr) as ?isKEmpty) {
    <http://example.org/ApplicationSchema#IExactGeom> geo:asWKT ?i_wkt .
    BIND(geof:isEmpty(?i_wkt) AS ?isIEmptyr)
     <http://example.org/ApplicationSchema#KExactGeom> geo:asWKT ?k_wkt .
    BIND(geof:isEmpty(?k_wkt) AS ?isKEmptyr)
}
"""),["i_wkt","k_wkt"],None)

LINESTRING EMPTY
True
POLYGON ((-77.089005 38.913574, -77.029953 38.913574, -77.029953 38.886321, -77.089005 38.886321, -77.089005 38.913574))
False


Variable,Value
i_wkt,LINESTRING EMPTY
k_wkt,"Polygon((-77.089005 38.913574,-77.029953 38.913574,-77.029953 38.886321,-77.089005 38.886321,-77.089005 38.913574))"
isIEmpty,true
isKEmpty,false


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:isMeasured](http://www.opengis.net/def/function/geosparql/isMeasured) Function

In [9]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?isEMeasured ?isLMeasured {
    BIND("POINT (1 2)"^^geo:wktLiteral AS ?e_wkt)
    BIND("POINT M (1 2 3)"^^geo:wktLiteral AS ?l_wkt)
    BIND(geof:isMeasured(?e_wkt) AS ?isEMeasured)
    BIND(geof:isMeasured(?l_wkt) AS ?isLMeasured)
}
"""),["e_wkt","l_wkt"],None)

Variable,Value
isEMeasured,false
isLMeasured,true


Map(center=[-83.2, 34.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:isSimple](http://www.opengis.net/def/function/geosparql/isSimple) Function

In [19]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?a_wkt ?o_wkt ?isASimple ?isOSimple {
<http://example.org/ApplicationSchema#AExactGeom> geo:asWKT ?a_wkt .
BIND(geof:isSimple(?a_wkt) AS ?isASimple)
<http://example.org/ApplicationSchema#OExactGeom> geo:asWKT ?o_wkt .
BIND(geof:isSimple(?o_wkt) AS ?isOSimple)
}
"""),["a_wkt","o_wkt"],None)

True
The Lit: true
False
The Lit: false


Variable,Value
a_wkt,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
isASimple,true
o_wkt,"Polygon ((2.47217 47.52403, 2.40693 47.49713, 2.44607 47.43679, 2.57104 47.48553, 2.39251 47.44793, 2.47217 47.52403))"
isOSimple,false


Map(center=[47.47714370265989, 2.4756124648711317], controls=(ZoomControl(options=['position', 'zoom_in_text',…

### [geof:maxX](http://www.opengis.net/def/function/geosparql/maxX) Function

Retrieves the maximum X coordinate of a GeoSPARQL geometry.
The X coordinate axis is defined by the CRS of the respective geometry.

In [10]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?maxX
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:maxX(?aLiteral) as ?maxX)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
maxX,-83.2


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:maxY](http://www.opengis.net/def/function/geosparql/maxY) Function

Retrieves the maximum Y coordinate of a GeoSPARQL geometry. The Y coordinate axis is defined by the CRS of the respective geometry.

In [11]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?maxY
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:maxY(?aLiteral) as ?maxY)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
maxY,34.5


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:maxZ](http://www.opengis.net/def/function/geosparql/maxZ) Function

Retrieves the maximum Z coordinate of a GeoSPARQL geometry. The Z coordinate axis is defined by the CRS of the respective geometry.

In [12]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?maxZ
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:maxZ(?literal) AS ?maxZ)
}
"""),["literal"],None)

Variable,Value
literal,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
maxZ,2.0


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### [geof:metricArea](http://www.opengis.net/def/function/geosparql/metricArea) Function

Returns the area of a GeoSPARQL geometry in squaremeters.

In [13]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?marea
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricArea(?aLiteral) as ?marea)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
marea,2400116828.6431704


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:metricBuffer](http://www.opengis.net/def/function/geosparql/metricBuffer) Function

In [14]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?buffer
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricBuffer(?aLiteral,"5.0"^^xsd:double) as ?buffer)
}
"""),["aLiteral"],None)

GEOMTOLIT: POLYGON ((-9306314.43031767 4042237.4996761675, -9306314.43031767 4096139.0404472337, -9306314.334244072 4096140.0158988438, -9306314.049715333 4096140.9538643956, -9306313.587665731 4096141.818298399, -9306312.965851575 4096142.5759811397, -9306312.208168834 4096143.197795295, -9306311.343734832 4096143.6598448963, -9306310.405769281 4096143.9443736356, -9306309.43031767 4096144.0404472337, -9261781.634000361 4096144.0404472337, -9261780.65854875 4096143.9443736356, -9261779.720583199 4096143.6598448963, -9261778.856149197 4096143.197795295, -9261778.098466456 4096142.5759811397, -9261777.4766523 4096141.818298399, -9261777.014602698 4096140.9538643956, -9261776.730073959 4096140.0158988438, -9261776.634000361 4096139.0404472337, -9261776.634000361 4042237.4996761675, -9261776.730073959 4042236.5242245574, -9261777.014602698 4042235.5862590056, -9261777.4766523 4042234.721825002, -9261778.098466456 4042233.9641422615, -9261778.856149197 4042233.342328106, -9261779.720583199

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
buffer,"POLYGON ((-9306314.43031767 4042237.4996761675, -9306314.43031767 4096139.0404472337, -9306314.334244072 4096140.0158988438, -9306314.049715333 4096140.9538643956, -9306313.587665731 4096141.818298399, -9306312.965851575 4096142.5759811397, -9306312.208168834 4096143.197795295, -9306311.343734832 4096143.6598448963, -9306310.405769281 4096143.9443736356, -9306309.43031767 4096144.0404472337, -9261781.634000361 4096144.0404472337, -9261780.65854875 4096143.9443736356, -9261779.720583199 4096143.6598448963, -9261778.856149197 4096143.197795295, -9261778.098466456 4096142.5759811397, -9261777.4766523 4096141.818298399, -9261777.014602698 4096140.9538643956, -9261776.730073959 4096140.0158988438, -9261776.634000361 4096139.0404472337, -9261776.634000361 4042237.4996761675, -9261776.730073959 4042236.5242245574, -9261777.014602698 4042235.5862590056, -9261777.4766523 4042234.721825002, -9261778.098466456 4042233.9641422615, -9261778.856149197 4042233.342328106, -9261779.720583199 4042232.880278505, -9261780.65854875 4042232.5957497656, -9261781.634000361 4042232.4996761675, -9306309.43031767 4042232.4996761675, -9306310.405769281 4042232.5957497656, -9306311.343734832 4042232.880278505, -9306312.208168834 4042233.342328106, -9306312.965851575 4042233.9641422615, -9306313.587665731 4042234.721825002, -9306314.049715333 4042235.5862590056, -9306314.334244072 4042236.5242245574, -9306314.43031767 4042237.4996761675))"


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:metricDistance](http://www.opengis.net/def/function/geosparql/metricDistance) Function

This function returns a distance between GeoSPARQL geometries in meters.

In [20]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?cLiteral ?fLiteral ?mdistance
WHERE {
  my:C geo:hasDefaultGeometry ?cGeom .
  ?cGeom geo:asWKT ?cLiteral .
  my:F geo:hasDefaultGeometry ?fGeom .
  ?fGeom geo:asWKT ?fLiteral .
  BIND (geof:metricDistance(?cLiteral, ?fLiteral) as ?mdistance)
}
"""),["cLiteral","fLiteral"],None)

Variable,Value
cLiteral,"Polygon((-83.2 34.3, -83.0 34.3, -83.0 34.5, -83.2 34.5, -83.2 34.3))"
fLiteral,Point(-83.4 34.4)
mdistance,22263.89815865457


Map(center=[34.4, -83.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

### [geof:metricLength](http://www.opengis.net/def/function/geosparql/metricLength) Function

In [21]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?mlength
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:metricLength(?aLiteral) as ?mlength)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
mlength,196858.6741767507


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minX](http://www.opengis.net/def/function/geosparql/minX) Function

Retrieves the minimum X coordinate of a GeoSPARQL geometry. The X coordinate axis is defined by the CRS of the respective geometry.

In [15]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?minX
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minX(?aLiteral) as ?minX)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
minX,-83.6


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minY](http://www.opengis.net/def/function/geosparql/minY) Function

Retrieves the minimum Y coordinate of a GeoSPARQL geometry. The Y coordinate axis is defined by the CRS of the respective geometry.

In [16]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?aLiteral ?minY
WHERE {
  my:A geo:hasDefaultGeometry ?aGeom .
  ?aGeom geo:asWKT ?aLiteral .
  BIND (geof:minY(?aLiteral) as ?minY)
}
"""),["aLiteral"],None)

Variable,Value
aLiteral,"Polygon((-83.6 34.1, -83.2 34.1, -83.2 34.5, -83.6 34.5, -83.6 34.1))"
minY,34.1


Map(center=[34.300000000000004, -83.39999999999999], controls=(ZoomControl(options=['position', 'zoom_in_text'…

### [geof:minZ](http://www.opengis.net/def/function/geosparql/minZ) Function

Retrieves the minimum Z coordinate of a GeoSPARQL geometry. The Z coordinate axis is defined by the CRS of the respective geometry.

In [17]:
processQueryResults(g.query(
"""
PREFIX my: <http://example.org/ApplicationSchema#>
PREFIX geo: <"""+str(GEO)+""">
PREFIX geof: <"""+str(GEOF)+""">
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
SELECT ?literal ?minZ
WHERE {
  my:NExactGeom geo:asWKT ?literal .
  BIND(geof:minZ(?literal) AS ?minZ)
}
"""),["literal"],None)

[-77.089005, 38.913574, 1.0]
[-77.029953, 38.913574, 2.0]
[-77.029953, 38.886321, 2.0]
[-77.089005, 38.886321, 1.0]
[-77.089005, 38.913574, 1.0]


Variable,Value
literal,"Polygon Z((-77.089005 38.913574 1,-77.029953 38.913574 2,-77.029953 38.886321 2,-77.089005 38.886321 1,-77.089005 38.913574 1))"
minZ,1.0


Map(center=[38.8999475, -77.05947900000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…